In [1]:
!pip install findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Funciones para trabajo con Strings").getOrCreate()

In [5]:
data = spark.read.parquet("./data/data.parquet")

In [6]:
data.printSchema()

root
 |-- nombre: string (nullable = true)



In [7]:
data.show()

+-------+
| nombre|
+-------+
| Spark |
+-------+



In [9]:
# podemos ver que la palabra spark tiene espacios como al principio y como al final

from pyspark.sql.functions import ltrim, rtrim, trim

In [10]:
data.select(
    ltrim("nombre").alias("ltrim"),
    rtrim("nombre").alias("rtrim"),
    trim("nombre").alias("trim")
).show()

+------+------+-----+
| ltrim| rtrim| trim|
+------+------+-----+
|Spark | Spark|Spark|
+------+------+-----+



## Rellenar un String con una longitud Fija

In [11]:
from pyspark.sql.functions import lpad, rpad

In [13]:
from pyspark.sql.functions import col, lpad, rpad, trim

data.select(
    trim(col("nombre")).alias("trim")
).select(
    lpad(col("trim"), 8, "-").alias("lpad"),
    rpad(col("trim"), 8, "=").alias("rpad")
).show()

+--------+--------+
|    lpad|    rpad|
+--------+--------+
|---Spark|Spark===|
+--------+--------+



Para ver el funcionamiento de las siguientes funciones vamos a crear un Dataframe nuevo

In [14]:
df1 = spark.createDataFrame([("spark", "es", "maravilloso")], ["sujeto" , "verbo", "adjetivo"])

In [15]:
df1.show()

+------+-----+-----------+
|sujeto|verbo|   adjetivo|
+------+-----+-----------+
| spark|   es|maravilloso|
+------+-----+-----------+



In [16]:
# vamos a haceru na concatenacion a trbajar con mayusculas y minusculas y hacer el reverso de un String

In [18]:
from pyspark.sql.functions import concat_ws, lower, upper, initcap, reverse

In [20]:
df1.select(
    concat_ws(" ", col("sujeto"), col("verbo"), col("adjetivo")).alias("Frase")
).select(
    lower(col("Frase")).alias("minusculas"),
    upper(col("Frase")).alias("mayusculas"),
    initcap(col("Frase")).alias("primera_mayuscula"),
    reverse(col("Frase")).alias("reverso")
).show()

+--------------------+--------------------+--------------------+--------------------+
|          minusculas|          mayusculas|   primera_mayuscula|             reverso|
+--------------------+--------------------+--------------------+--------------------+
|spark es maravilloso|SPARK ES MARAVILLOSO|Spark Es Maravilloso|osollivaram se kraps|
+--------------------+--------------------+--------------------+--------------------+



## Regex_replace

In [24]:
# nos permite remplazar un patron por algo que nosotros le digamos

from pyspark.sql.functions import regexp_replace

In [25]:
df2 = spark.createDataFrame([("voy a casa por mis llaves",)], ["frase"])

In [27]:
df2.show(truncate=False)

+-------------------------+
|frase                    |
+-------------------------+
|voy a casa por mis llaves|
+-------------------------+



In [28]:
df2.select(
    regexp_replace(col("frase"), "voy|por", "ir").alias("frase_reemplazada")
).show(truncate=False
)

+-----------------------+
|frase_reemplazada      |
+-----------------------+
|ir a casa ir mis llaves|
+-----------------------+



In [29]:
# si nos fijamos donde encontro la palabra voy y por | la sustituyo por ir los patrones pueden ser tan complejos como el caso especifico de ese momento